# 🚀 Aero Browse — VLA Browser Agent Training Pipeline

This notebook runs the complete fine-tuning, merging, quantizing, and uploading pipeline for the Aero Browse VLA agent based on **SmolVLM-Instruct** (2.2B) and the **osunlp/Multimodal-Mind2Web** dataset.

It has been fully optimized to run on both Kaggle and cloud VM instances (like Thunder Compute) by avoiding OOMs, running filtering sequentially before model loading, using non-paged optimizers, splitting the dataset into train/validation sets for real-time loss tracking, and auto-patching tokenizer configuration bugs.

## 📋 0. Configuration

Configure your paths, model options, and hyperparameters below.

In [ ]:
# ═══════════════════════════════════════════════════
# CONFIGURATION — Edit these before running
# ═══════════════════════════════════════════════════

# Base Model (You can also pass your own previously uploaded HF model here to fine-tune it further!)
BASE_MODEL = "HuggingFaceTB/SmolVLM-Instruct"

# Training Hyperparameters
BATCH_SIZE = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_STEPS = 500
LORA_R = 16
LORA_ALPHA = 32
MAX_SEQ_LENGTH = 1024
SUBSET_SIZE = None  # Set to an integer for a subset (e.g. 1000) or None for full dataset

# Output directories
OUTPUT_DIR = "./aero-browse-sft-output"
ADAPTER_DIR = "./aero-browse_lora_adapter"
MERGED_DIR = "./aero-browse-merged"
GGUF_DIR = "./aero-browse-gguf"

# W&B Configurations
WANDB_PROJECT = "aero-browse"
WANDB_RUN_NAME = "smolvlm-lora-mind2web"

# Hugging Face Hub (Set to your credentials)
HF_REPO_ID = None      # e.g., "username/aero-browse-vla"
HF_GGUF_REPO_ID = None # e.g., "username/aero-browse-vla-gguf"

# Pipeline Control
SKIP_GGUF = False
SKIP_PUSH = True  # Set to False to push to Hugging Face

print("✅ Configuration loaded.")

## 🛠️ 1. Setup Environment & API Keys

In [ ]:
# Install dependencies with CUDA 12 support
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers peft trl accelerate bitsandbytes datasets wandb sentencepiece Pillow tqdm

import os
import sys
import torch

# Try loading from Kaggle vault
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("✅ Loaded secrets from Kaggle vault.")
except Exception:
    print("⚠️  Kaggle secrets not found. Using environment variables instead.")
    
hf_token = os.environ.get("HF_TOKEN")
wandb_api_key = os.environ.get("WANDB_API_KEY")

if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA-enabled GPU is required for training.")
    
print(f"🚀 Device: {torch.cuda.get_device_name(0)}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 🤗 2. W&B and Hugging Face Authentication

In [ ]:
import wandb
from huggingface_hub import login as hf_login

if hf_token:
    try:
        hf_login(token=hf_token, add_to_git_credential=False)
        print("✅ Logged in to Hugging Face Hub.")
    except Exception as e:
        print(f"⚠️  HF login failed: {e}")
        hf_token = None
else:
    print("⚠️  No HF_TOKEN found. Pushing to HF Hub will be skipped.")
    SKIP_PUSH = True
    
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "base_model": BASE_MODEL,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "max_steps": MAX_STEPS,
    }
)
print("✅ W&B initialized.")

## 📝 3. Load Tokenizer & Inject Custom Action Tokens

In [ ]:
from transformers import AutoProcessor, AutoTokenizer, Idefics3ImageProcessor, Idefics3Processor

# Custom coordinate and action tokens
class VLATokenizerConfig:
    def __init__(self):
        self.action_tokens = ["<click>", "<type>", "<hover>", "<terminate>"]
        self.coord_tokens = [f"<loc_{i}>" for i in range(1001)]
        self.all_custom_tokens = self.action_tokens + self.coord_tokens
        
vla_config = VLATokenizerConfig()

# Resolve processor loading to bypass AutoProcessor config detection bugs
processor = None
try:
    print("      Attempting to load Idefics3ImageProcessor manually...")
    image_processor = Idefics3ImageProcessor.from_pretrained(BASE_MODEL, token=hf_token)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token)
    processor = Idefics3Processor(image_processor=image_processor, tokenizer=tokenizer)
    print("      Successfully loaded processor via manual construction.")
except Exception as e:
    print(f"      Manual load failed: {e}. Falling back to AutoProcessor.")
    processor = AutoProcessor.from_pretrained(BASE_MODEL, token=hf_token)
    
tokenizer = processor.tokenizer
num_added = tokenizer.add_tokens(vla_config.all_custom_tokens)
print(f"✅ Added {num_added} custom coordinate & action tokens to tokenizer.")

## 📊 4. Load & Filter Multimodal-Mind2Web Dataset

We load and filter the dataset sequentially **before** loading the base model. This keeps the memory usage low during multiprocessing/forking.

In [ ]:
import json
from datasets import load_dataset
from PIL import Image
from torch.utils.data import Dataset

# Custom Dataset class
class BrowserAgentDataset(Dataset):
    def __init__(self, hf_dataset, processor):
        self.dataset = hf_dataset
        self.processor = processor
        self.vla_config = vla_config
        
    def __len__(self):
        return len(self.dataset)
        
    def __getitem__(self, idx):
        row = self.dataset[idx]
        user_goal = row["confirmed_task"]
        op = row["operation"]
        if isinstance(op, str):
            op = json.loads(op)
        op_type = op.get("op", "CLICK")
        value = op.get("value", "")
        
        candidates = row["pos_candidates"]
        target_cand = None
        for c_str in candidates:
            c = json.loads(c_str) if isinstance(c_str, str) else c_str
            if c.get("is_original_target"):
                target_cand = c
                break
        if not target_cand:
            for c_str in candidates:
                c = json.loads(c_str) if isinstance(c_str, str) else c_str
                if c.get("is_top_level_target"):
                    target_cand = c
                    break
        if not target_cand and candidates:
            c_str = candidates[0]
            target_cand = json.loads(c_str) if isinstance(c_str, str) else c_str
            
        screenshot = row["screenshot"]
        W, H = screenshot.size
        H_crop = 1024
        scaled_x, scaled_y = 500, 500
        image = screenshot
        has_valid_coords = False
        
        if target_cand:
            attrs = target_cand.get("attributes", {})
            if isinstance(attrs, str):
                attrs = json.loads(attrs)
            bbox_str = attrs.get("bounding_box_rect")
            if bbox_str:
                try:
                    parts = [float(x) for x in bbox_str.split(",")]
                    if len(parts) == 4:
                        x, y, w, h = parts
                        cx = x + w / 2
                        cy = y + h / 2
                        has_valid_coords = True
                except:
                    pass
                    
        if H > H_crop:
            if has_valid_coords:
                y_start = max(0, min(H - H_crop, int(cy - H_crop / 2)))
                image = screenshot.crop((0, y_start, W, y_start + H_crop)).copy()
                new_cy = cy - y_start
            else:
                image = screenshot.crop((0, 0, W, H_crop)).copy()
                new_cy = 500
            actual_crop_height = H_crop
        else:
            image = screenshot.copy()
            new_cy = cy if has_valid_coords else 500
            actual_crop_height = H
            
        if has_valid_coords:
            scaled_x = max(0, min(1000, int((cx / W) * 1000)))
            scaled_y = max(0, min(1000, int((new_cy / actual_crop_height) * 1000)))
            
        action = "type" if op_type == "TYPE" else "click"
        text_content = value if op_type == "TYPE" else None
        
        # Format action string
        if action == "click":
            target_action_str = f"<{action}><loc_{scaled_x}><loc_{scaled_y}>"
        else:
            target_action_str = f"<{action}><loc_{scaled_x}><loc_{scaled_y}>{text_content}"
        target_action_str += "<terminate>"
        
        prompt = f"<image>\nUser Objective: {user_goal}\nNext Action:"
        full_text = prompt + target_action_str
        
        if image.mode != "RGB":
            image = image.convert("RGB")
            
        # Resize image to max 768px longest edge to avoid GPU OOM
        max_img_size = 768
        if max(image.size) > max_img_size:
            image.thumbnail((max_img_size, max_img_size), Image.Resampling.LANCZOS)
            
        inputs = self.processor(text=full_text, images=image, return_tensors="pt")
        input_ids = inputs["input_ids"].squeeze(0)
        labels = input_ids.clone()
        
        target_inputs = self.processor.tokenizer(target_action_str, add_special_tokens=False)
        target_len = len(target_inputs["input_ids"])
        prompt_len = max(0, input_ids.shape[0] - target_len)
        labels[:prompt_len] = -100
        
        return {
            "input_ids": input_ids,
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "pixel_attention_mask": inputs["pixel_attention_mask"].squeeze(0),
            "labels": labels
        }
        
print("      Loading Multimodal-Mind2Web dataset...")
hf_dataset = load_dataset("osunlp/Multimodal-Mind2Web", split="train")

def has_bbox_filter(example):
    try:
        candidates = example["pos_candidates"]
        if not candidates:
            return False
        for c_str in candidates:
            c = json.loads(c_str) if isinstance(c_str, str) else c_str
            attrs = c.get("attributes", {})
            if isinstance(attrs, str):
                attrs = json.loads(attrs)
            if attrs.get("bounding_box_rect"):
                return True
        return False
    except:
        return False
        
print("      Filtering dataset sequentially (low memory)...")
filtered_dataset = hf_dataset.filter(has_bbox_filter)
print(f"✅ Filtered dataset size: {len(filtered_dataset)} samples")

if SUBSET_SIZE is not None and SUBSET_SIZE < len(filtered_dataset):
    print(f"      Selecting a subset of {SUBSET_SIZE} samples...")
    filtered_dataset = filtered_dataset.select(range(SUBSET_SIZE))
    
# Split into 95% train and 5% validation for tracking validation loss
print("      Splitting dataset into 95% train and 5% validation...")
split_ds = filtered_dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = BrowserAgentDataset(split_ds["train"], processor)
eval_dataset = BrowserAgentDataset(split_ds["test"], processor)
print(f"      Train samples: {len(train_dataset)}, Validation samples: {len(eval_dataset)}")

## 🤖 5. Load Base Model in 4-bit (NF4) with SDPA

In [ ]:
from transformers import BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except ImportError:
    from transformers import AutoModelForVision2Seq
    
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("      Loading base model...")
model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    _attn_implementation="sdpa",
    token=hf_token,
)
model.resize_token_embeddings(len(tokenizer))
print("✅ Base model loaded successfully.")

## 🔗 6. Apply LoRA Adapter Config

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["embed_tokens", "lm_head"],
    task_type="CAUSAL_LM",
    ensure_weight_tying=True,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("✅ LoRA adapter wrapped.")

## 🚀 7. SFT Fine-Tuning

Runs the SFT trainer. We use `adamw_8bit` here to avoid UVM (Unified Virtual Memory) crash on Thunder Compute prototyping instances.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    max_steps=MAX_STEPS,
    optim="adamw_8bit",  # Safe non-paged optimizer for prototyping VM mode
    bf16=True,
    remove_unused_columns=False,
    save_strategy="steps",
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    max_length=MAX_SEQ_LENGTH,
    report_to="wandb",
    warmup_steps=20,
    save_total_limit=1,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

def collate_fn(examples):
    input_ids = [e["input_ids"] for e in examples]
    attention_mask = [e["attention_mask"] for e in examples]
    labels = [e["labels"] for e in examples]
    pixel_values = [e["pixel_values"] for e in examples]
    pixel_attention_mask = [e["pixel_attention_mask"] for e in examples]
    
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    
    max_patches = max(p.shape[0] for p in pixel_values)
    padded_pixel_values = []
    padded_pixel_attention_mask = []
    
    for p, p_mask in zip(pixel_values, pixel_attention_mask):
        num_patches = p.shape[0]
        padded_pixel_values.append(torch.nn.functional.pad(p, (0,0,0,0,0,0,0,max_patches - num_patches), value=0))
        padded_pixel_attention_mask.append(torch.nn.functional.pad(p_mask, (0,0,0,max_patches - num_patches), value=0))
        
    pixel_values = torch.stack(padded_pixel_values)
    pixel_attention_mask = torch.stack(padded_pixel_attention_mask)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": pixel_values,
        "pixel_attention_mask": pixel_attention_mask
    }
    
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=collate_fn,
)

print("      Starting training...")
trainer.train()

# Save adapter
print(f"      Saving adapter to {ADAPTER_DIR}...")
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("✅ Adapter saved.")

## 🔗 8. Merge LoRA Adapter Weights on CPU

Merges LoRA adapter into the base model weights on CPU to avoid GPU OOM during merging.

In [ ]:
import gc
from peft import PeftModel

# Clean up GPU memory
del model, trainer
gc.collect()
torch.cuda.empty_cache()

print("      Loading base model on CPU...")
base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    token=hf_token,
)
base_model.resize_token_embeddings(len(tokenizer))

print("      Merging weights...")
model_with_lora = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = model_with_lora.merge_and_unload()

print(f"      Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("✅ LoRA weights merged successfully.")

del base_model, model_with_lora, merged_model
gc.collect()

## ⚙️ 9. Tokenizer Patching & GGUF Conversion

We patch `tokenizer_config.json` inside the merged model directory to prevent `llama.cpp`'s `extra_special_tokens` crash. Then, we clone `llama.cpp` and convert the model to FP16 GGUF.

In [ ]:
import subprocess

# Patch tokenizer configs to prevent extra_special_tokens crash in llama.cpp
print("      Patching tokenizer configs to prevent extra_special_tokens crash...")
for target_dir in [MERGED_DIR, ADAPTER_DIR]:
    tok_config_path = os.path.join(target_dir, "tokenizer_config.json")
    if os.path.exists(tok_config_path):
        try:
            with open(tok_config_path, "r", encoding="utf-8") as f:
                tok_config = json.load(f)
            if "extra_special_tokens" in tok_config:
                del tok_config["extra_special_tokens"]
                with open(tok_config_path, "w", encoding="utf-8") as f:
                    json.dump(tok_config, f, indent=2, ensure_ascii=False)
                print(f"      Successfully patched {tok_config_path}")
        except Exception as e:
            print(f"      Warning: Failed to patch {tok_config_path}: {e}")
            
if not SKIP_GGUF:
    print("      Cloning llama.cpp...")
    llama_cpp_dir = "./llama.cpp"
    if not os.path.exists(llama_cpp_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git", llama_cpp_dir], check=True)
        
    print("      Installing llama.cpp requirements...")
    subprocess.run(["pip", "install", "-q", "-r", os.path.join(llama_cpp_dir, "requirements.txt")], check=True)
    
    os.makedirs(GGUF_DIR, exist_ok=True)
    convert_script = os.path.join(llama_cpp_dir, "convert_hf_to_gguf.py")
    gguf_fp16_path = os.path.join(GGUF_DIR, "aero-browse-f16.gguf")
    
    print("      Converting to FP16 GGUF...")
    res = subprocess.run(["python", convert_script, MERGED_DIR, "--outfile", gguf_fp16_path, "--outtype", "f16"], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"⚠️  GGUF FP16 conversion failed:\n{res.stderr}")
    else:
        print(f"✅ FP16 GGUF saved at {gguf_fp16_path}")

## 🤗 10. Upload to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

if not SKIP_PUSH and HF_REPO_ID:
    print("      Uploading artifacts to HF Hub...")
    api = HfApi()
    
    # 1. Merged model
    print(f"      Uploading merged model to hf.co/{HF_REPO_ID}...")
    api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO_ID, commit_message="Upload Aero Browse VLA - Merged weights")
    
    # 2. Adapter
    adapter_repo = f"{HF_REPO_ID}-lora"
    print(f"      Uploading adapter to hf.co/{adapter_repo}...")
    api.create_repo(adapter_repo, exist_ok=True, private=False)
    api.upload_folder(folder_path=ADAPTER_DIR, repo_id=adapter_repo, commit_message="Upload Aero Browse VLA - LoRA adapter")
    
    # 3. GGUF
    if not SKIP_GGUF and HF_GGUF_REPO_ID and os.path.exists(GGUF_DIR):
        print(f"      Uploading GGUF models to hf.co/{HF_GGUF_REPO_ID}...")
        api.create_repo(HF_GGUF_REPO_ID, exist_ok=True, private=False)
        api.upload_folder(folder_path=GGUF_DIR, repo_id=HF_GGUF_REPO_ID, commit_message="Upload Aero Browse VLA - GGUF models")
        
    print("✅ HF Hub upload complete.")